In [1]:
# Basic libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Text preprocessing
import re
import string
from sklearn.model_selection import train_test_split

# Tokenization
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout

# Utils
from tensorflow.keras.utils import to_categorical

In [2]:
# Load dataset
df = pd.read_csv("all_complaints.csv")

# Show first 5 rows
df.head()

,complaint_text,department
0,No water supply since morning in my area.,Water Department
1,Water pipe burst near the main road causing fl...,Water Department
2,Tap water has a foul smell for the past few days.,Water Department
3,Water leakage observed near the park.,Water Department
4,Low water pressure in our apartment building.,Water Department


In [3]:
# Text cleaning function
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>+', '', text)
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'\w*\d\w*', '', text)
    return text

# Apply cleaning
df['clean_text'] = df['complaint_text'].apply(clean_text)

# Preview
df[['complaint_text', 'clean_text']].head()

,complaint_text,clean_text
0,No water supply since morning in my area.,no water supply since morning in my area
1,Water pipe burst near the main road causing fl...,water pipe burst near the main road causing fl...
2,Tap water has a foul smell for the past few days.,tap water has a foul smell for the past few days
3,Water leakage observed near the park.,water leakage observed near the park
4,Low water pressure in our apartment building.,low water pressure in our apartment building


In [4]:
# Convert labels to numbers
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['department'])

# Features and labels
X = df['clean_text']
y = df['label']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Check
print("Sample labels:", df[['department', 'label']].drop_duplicates())
print("Train size:", len(X_train))
print("Test size:", len(X_test))

Sample labels:                   department  label
0           Water Department      4
102   Electricity Department      0
203  Public Works Department      2
303    Sanitation Department      3
404        Health Department      1
Train size: 405
Test size: 102


In [5]:
# Parameters
max_words = 10000   # vocabulary size
max_len = 100       # max sequence length

# Tokenizer
tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

# Convert text to sequences
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# Padding
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post')

# Check shape
print("Train shape:", X_train_pad.shape)
print("Test shape:", X_test_pad.shape)

Train shape: (405, 100)
Test shape: (102, 100)


In [8]:
import urllib.request
import zipfile
import os

# Download GloVe
url = "http://nlp.stanford.edu/data/glove.6B.zip"
zip_path = "glove.6B.zip"

if not os.path.exists(zip_path):
    print("Downloading GloVe...")
    urllib.request.urlretrieve(url, zip_path)

# Extract
if not os.path.exists("glove.6B.100d.txt"):
    print("Extracting...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall()

print("Done. Files ready!")

Extracting...
Done. Files ready!


In [9]:
embedding_index = {}

with open("glove.6B.100d.txt", encoding="utf8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype="float32")
        embedding_index[word] = vector

print("Total word vectors:", len(embedding_index))

Total word vectors: 400000


In [10]:
embedding_dim = 100

word_index = tokenizer.word_index
vocab_size = min(len(word_index) + 1, 10000)

embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, i in word_index.items():
    if i >= vocab_size:
        continue
    vector = embedding_index.get(word)
    if vector is not None:
        embedding_matrix[i] = vector

print("Embedding matrix shape:", embedding_matrix.shape)

Embedding matrix shape: (680, 100)


In [11]:
num_classes = len(df['label'].unique())

model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        input_length=max_len,
        trainable=False
    ),
    
    Bidirectional(LSTM(128, return_sequences=False)),
    Dropout(0.5),
    
    Dense(64, activation='relu'),
    Dropout(0.3),
    
    Dense(num_classes, activation='softmax')
])

model.summary()

c:\Users\niraj\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │        68,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 68,000 (265.62 KB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 68,000 (265.62 KB)

In [12]:
# Compile
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

# Train
history = model.fit(
    X_train_pad,
    y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test_pad, y_test)
)

Epoch 1/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 8s 116ms/step - accuracy: 0.2353 - loss: 1.5968 - val_accuracy: 0.5196 - val_loss: 1.4793
Epoch 2/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 92ms/step - accuracy: 0.4919 - loss: 1.4237 - val_accuracy: 0.5980 - val_loss: 1.3005
Epoch 3/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 84ms/step - accuracy: 0.6383 - loss: 1.1154 - val_accuracy: 0.6373 - val_loss: 1.0010
Epoch 4/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 85ms/step - accuracy: 0.7112 - loss: 0.8656 - val_accuracy: 0.6667 - val_loss: 0.8061
Epoch 5/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 76ms/step - accuracy: 0.7390 - loss: 0.6989 - val_accuracy: 0.7353 - val_loss: 0.6657
Epoch 6/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 78ms/step - accuracy: 0.7995 - loss: 0.5576 - val_accuracy: 0.7451 - val_loss: 0.6592
Epoch 7/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 76ms/step - accuracy: 0.8253 - loss: 0.4878 - val_accuracy: 0.7843 - val_loss: 0.6274
Epoch 8/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 95ms/step - accuracy: 0.8505 - loss: 0.4089 - val_accuracy: 0.8039 - 

In [13]:
def predict_text(text):
    # Clean text
    text = clean_text(text)
    
    # Convert to sequence
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=max_len, padding='post')
    
    # Predict
    pred = model.predict(pad)
    label_index = np.argmax(pred)
    
    # Convert back to label
    label = label_encoder.inverse_transform([label_index])[0]
    
    return label

# Test example
print(predict_text("Water is not coming in my house"))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 318ms/step
Water Department


In [14]:
# Save model
model.save("bilstm_glove_model.h5")

# Save tokenizer
import pickle
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

# Save label encoder
with open("label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

print("All files saved successfully ✅")

All files saved successfully ✅


In [15]:
def predict_text(text):
    # Clean text
    text = clean_text(text)
    
    # Convert to sequence
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=max_len, padding='post')
    
    # Predict
    pred = model.predict(pad)
    label_index = np.argmax(pred)
    
    # Convert back to label
    label = label_encoder.inverse_transform([label_index])[0]
    
    return label

# Test example
print(predict_text("In my area there are many potholes"))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
Public Works Department


In [18]:
import numpy as np
import pickle
import re
import string
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Load model
model = load_model("bilstm_glove_model.h5")

# Load tokenizer
with open("tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)

# Load label encoder
with open("label_encoder.pkl", "rb") as f:
    label_encoder = pickle.load(f)

max_len = 100

# Clean function
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>+', '', text)
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'\w*\d\w*', '', text)
    return text

# Prediction function
def predict_text(text):
    text = clean_text(text)
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=max_len, padding='post')
    
    pred = model.predict(pad)
    label_index = np.argmax(pred)
    label = label_encoder.inverse_transform([label_index])[0]
    
    return label

# Test
print(predict_text("power is not coming in my house"))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 320ms/step
Electricity Department
